# Compute Frugality as Internalized Value — Analysis Notebook

Run this after `run_pipeline.py` has completed.
Walks through all three tests interactively.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

OUTPUT_DIR = Path('../outputs')

## 1. Reward Curve During Training

Check that the frugal model's reward increased and that it didn't collapse.

In [ ]:
# Load training logs if available (TRL logs to trainer_state.json)
for run in ['baseline', 'frugal_a30']:
    log_path = OUTPUT_DIR / run / 'trainer_state.json'
    if not log_path.exists():
        print(f'{run}: no trainer_state.json found')
        continue
    
    with open(log_path) as f:
        state = json.load(f)
    
    steps   = [e['step']   for e in state['log_history'] if 'reward' in e]
    rewards = [e['reward'] for e in state['log_history'] if 'reward' in e]
    
    plt.plot(steps, rewards, label=run)

plt.xlabel('Step')
plt.ylabel('Mean Reward')
plt.title('Training Reward Curves')
plt.legend()
plt.show()

## 2. Test 1: Generalization

Does the frugal model produce fewer tokens on OOD task types?

In [ ]:
# Load eval results
eval_dir = OUTPUT_DIR / 'eval_results'

records = []
for model_name in ['baseline', 'frugal']:
    path = eval_dir / f'{model_name}_results.json'
    if not path.exists():
        print(f'No results for {model_name}')
        continue
    with open(path) as f:
        results = json.load(f)
    for r in results:
        records.append(r)

df = pd.DataFrame(records)
df['is_ood'] = df['task_type'].isin(['pattern', 'logic'])

neutral_df = df[df['condition'] == 'neutral']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Token counts
sns.boxplot(
    data=neutral_df, x='task_type', y='n_tokens', hue='model_name',
    ax=axes[0], palette={'baseline': 'steelblue', 'frugal': 'tomato'}
)
axes[0].set_title('Response Length by Task Type')
axes[0].set_xlabel('Task Type')
axes[0].set_ylabel('Tokens Generated')

# Accuracy
acc = neutral_df.groupby(['model_name', 'task_type'])['correctness'].mean().reset_index()
sns.barplot(
    data=acc, x='task_type', y='correctness', hue='model_name',
    ax=axes[1], palette={'baseline': 'steelblue', 'frugal': 'tomato'}
)
axes[1].set_title('Accuracy by Task Type')
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

# Summary table
summary = neutral_df.groupby(['model_name', 'task_type']).agg(
    mean_tokens=('n_tokens', 'mean'),
    accuracy=('correctness', 'mean'),
    n=('n_tokens', 'count')
).round(2)
print(summary)

## 3. Test 2: Monitoring Invariance

Does the model behave differently when told it's being monitored?

In [ ]:
cond_df = df.groupby(['model_name', 'condition'])['n_tokens'].agg(
    mean='mean', sem=lambda x: x.std() / np.sqrt(len(x))
).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
conditions = ['monitored', 'neutral', 'unmonitored']
colors = {'baseline': 'steelblue', 'frugal': 'tomato'}

for model_name, color in colors.items():
    m_df = cond_df[cond_df['model_name'] == model_name].set_index('condition')
    means = [m_df.loc[c, 'mean'] if c in m_df.index else np.nan for c in conditions]
    sems  = [m_df.loc[c, 'sem']  if c in m_df.index else 0      for c in conditions]
    ax.errorbar(conditions, means, yerr=sems, label=model_name,
                color=color, linewidth=2, marker='o', markersize=8, capsize=5)

ax.set_title('Response Length by Monitoring Condition', fontsize=13)
ax.set_ylabel('Mean Tokens Generated')
ax.set_xlabel('Monitoring Condition')
ax.legend(title='Model')
ax.grid(True, alpha=0.3)
plt.show()

# Compute invariance score
print('\nMonitoring Invariance Scores:')
for model_name in ['baseline', 'frugal']:
    m = df[df['model_name'] == model_name].groupby('condition')['n_tokens'].mean()
    if all(c in m.index for c in ['monitored', 'unmonitored', 'neutral']):
        score = 1.0 - abs(m['monitored'] - m['unmonitored']) / (m['neutral'] + 1e-8)
        print(f'  {model_name}: {score:.3f} (1.0=perfectly invariant)')

## 4. Test 3: Residual Stream Probe AUROC by Layer

In [ ]:
probe_dir = OUTPUT_DIR / 'probe_results'
probe_data = {}

for model_name in ['baseline', 'frugal']:
    path = probe_dir / f'{model_name}_probe_results.json'
    if path.exists():
        with open(path) as f:
            # Keys are strings from JSON
            raw = json.load(f)
            probe_data[model_name] = {int(k): v for k, v in raw.items()}
    else:
        print(f'No probe results for {model_name}')

if probe_data:
    n_layers = max(max(d.keys()) for d in probe_data.values()) + 1
    layers = list(range(n_layers))

    fig, ax = plt.subplots(figsize=(12, 5))
    
    for model_name, color in [('baseline', 'steelblue'), ('frugal', 'tomato')]:
        if model_name not in probe_data:
            continue
        d = probe_data[model_name]
        aurocs = [d.get(l, {}).get('auroc', 0.5) for l in layers]
        stds   = [d.get(l, {}).get('std',   0.0) for l in layers]
        
        ax.plot(layers, aurocs, color=color, label=model_name, linewidth=2, marker='o', markersize=4)
        ax.fill_between(layers,
                        [a - s for a, s in zip(aurocs, stds)],
                        [a + s for a, s in zip(aurocs, stds)],
                        alpha=0.15, color=color)
    
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
    ax.set_xlabel('Layer Index', fontsize=12)
    ax.set_ylabel('Probe AUROC', fontsize=12)
    ax.set_title('"Will produce short output" — Probe AUROC by Layer', fontsize=13)
    ax.legend(fontsize=11)
    ax.set_ylim(0.4, 1.05)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Interpretation
    print('\nInterpretation:')
    print('If the frugal model shows higher AUROC (especially in earlier layers),')
    print('this suggests a stable pre-generation representation of the compute-efficiency value.')

## 5. Qualitative Examples

Side-by-side comparison of responses on the same prompt.

In [ ]:
# Sample some tasks and show baseline vs. frugal responses
baseline_df = pd.DataFrame(
    json.load(open(eval_dir / 'baseline_results.json'))
).query("condition == 'neutral'")

frugal_df = pd.DataFrame(
    json.load(open(eval_dir / 'frugal_results.json'))
).query("condition == 'neutral'")

merged = baseline_df.merge(
    frugal_df[['task_id', 'response', 'n_tokens', 'correctness']],
    on='task_id', suffixes=('_base', '_frugal')
)

# Show biggest token-count differences
merged['token_delta'] = merged['n_tokens_base'] - merged['n_tokens_frugal']
interesting = merged.nlargest(5, 'token_delta')

for _, row in interesting.iterrows():
    print(f"\n{'='*60}")
    print(f"Task: {row['task_id']} | {row['task_type']}")
    print(f"Prompt: {row['prompt'][:100]}...")
    print(f"\nBaseline ({row['n_tokens_base']} tokens, correct={row['correctness_base']:.0f}):")
    print(f"  {row['response_base'][:200]}")
    print(f"\nFrugal ({row['n_tokens_frugal']} tokens, correct={row['correctness_frugal']:.0f}):")
    print(f"  {row['response_frugal'][:200]}")